# Walkthrough

A walkthrough of the entire experiment for an eligible complex. Expects data under `/src/data/`:

- `/src/data/diffdock`: contains candidate poses per complex.
- `/src/data/posebusters`: contains protein files.

In [1]:
import os
import pandas as pd

import utils, filter, prepare, encode, confidence
import importlib
importlib.reload(utils)
importlib.reload(filter)
importlib.reload(prepare)
importlib.reload(encode)
importlib.reload(confidence)

<module 'confidence' from '/Users/mattgc/code/thesis-custom/research/sapt-preproc/src/confidence.py'>

In [2]:
SRC = os.getcwd()
DATA = os.path.join(SRC, 'data')
DIFFDOCK = os.path.join(DATA, "diffdock")
POSEBUSTERS = os.path.join(DATA, "posebusters")
OUT = os.path.join(SRC, 'out')
NAME = "v1_1_mm_unsize"
JOB = os.path.join(OUT, f"filter_{NAME}")
FILTERED = os.path.join(JOB, "filter.csv")
CHOSEN = os.path.join(JOB, "chosen.csv")

# Experiment Filter

## Preparation

Before the protein (monomer A) and poses (monomer Bs) are encoded, the complex must be prepared. Preparation loads the proteins and poses, verifies they are in scope, cleans them, fixes them, protonates both sides, minimises the poses, reverifies, then finally calculates the net charge and verifies that the number of electrons is even.

`filter` prepares all complexes DiffDock ran over and returns the generated artefacts for eligible ones.

In [3]:
# WARNING: takes several hours
# filter.run(name="filter_v1_1_mm")

In [4]:
filtered = pd.read_csv(FILTERED)
filtered.head()

,name,status,heavy_atoms,charge,electrons,poses,excluded,near_native,rejection
0,5SAK_ZRY,eligible,300.0,-6.0,2282.0,36,4.0,7,NaN
1,5SB2_1K2,eligible,297.0,-1.0,2306.0,40,0.0,40,NaN
2,5SD5_HWI,rejected,NaN,NaN,NaN,40,NaN,0,ligand carrying a group ionised at pH 7.4
3,6M2B_EZO,rejected,NaN,NaN,NaN,40,NaN,0,ligand carrying a group ionised at pH 7.4
4,6M73_FNR,rejected,NaN,NaN,NaN,40,NaN,0,ligand carrying a group ionised at pH 7.4


In [5]:
filtered["difficulty"] = filtered["near_native"] / filtered["poses"]
filtered = filtered[
    (filtered["status"] == "eligible") 
        & (filtered["difficulty"] < 0.5) 
        # & (-2 < df["charge"])
]
filtered = filtered.sort_values(by=["electrons"])[:12]
filtered

,name,status,heavy_atoms,charge,electrons,poses,excluded,near_native,rejection,difficulty
59,7LOE_Y84,eligible,147.0,0.0,1144.0,40,0.0,1,NaN,0.025000
46,7F5D_EUO,eligible,163.0,-3.0,1226.0,40,0.0,8,NaN,0.200000
149,7W05_GMP,eligible,183.0,-1.0,1364.0,39,1.0,5,NaN,0.128205
84,7OEO_V9Z,eligible,210.0,-1.0,1594.0,40,0.0,7,NaN,0.175000
88,7OSO_0V1,eligible,228.0,-1.0,1720.0,37,3.0,8,NaN,0.216216
161,7XFA_D9J,eligible,231.0,1.0,1760.0,27,12.0,4,NaN,0.148148
167,7Z2O_IAJ,eligible,251.0,0.0,1870.0,40,0.0,3,NaN,0.075000
102,7Q2B_M6H,eligible,268.0,-3.0,2038.0,37,3.0,1,NaN,0.027027
191,8DSC_NCA,eligible,274.0,1.0,2088.0,40,0.0,15,NaN,0.375000
78,7NGW_UAW,eligible,292.0,-1.0,2218.0,39,0.0,16,NaN,0.410256


In [6]:
filtered.to_csv(CHOSEN)

## DiffDock's confidence

In [7]:
chosen = pd.read_csv(CHOSEN)
chosen_complexes= list(chosen["name"])
chosen_complexes

['7LOE_Y84',
 '7F5D_EUO',
 '7W05_GMP',
 '7OEO_V9Z',
 '7OSO_0V1',
 '7XFA_D9J',
 '7Z2O_IAJ',
 '7Q2B_M6H',
 '8DSC_NCA',
 '7NGW_UAW',
 '5SAK_ZRY',
 '7PRM_81I']

In [29]:
confidence.run(
    complexes=chosen_complexes,
    python="/opt/anaconda3/envs/diffdock-mac/bin/python",
    name=NAME
)


/opt/anaconda3/envs/diffdock-mac/lib/python3.9/site-packages/torch/jit/_check.py:181: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "
/opt/anaconda3/envs/diffdock-mac/lib/python3.9/site-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


Scoring on cpu
Processing 1 of 1 batches (1 sequences)


@> 2389 atoms and 1 coordinate set(s) were parsed in 0.02s.
@> 2389 atoms and 1 coordinate set(s) were parsed in 0.01s.
/Users/mattgc/code/thesis-custom/research/sapt-preproc/src/diffdock/datasets/parse_chi.py:91: RuntimeWarning: invalid value encountered in cast
  Y = indices.astype(int)


[5SAK_ZRY] 36 poses, best -0.10
Processing 1 of 1 batches (1 sequences)


@> 905 atoms and 1 coordinate set(s) were parsed in 0.01s.
@> 905 atoms and 1 coordinate set(s) were parsed in 0.01s.


...

@> 3190 atoms and 1 coordinate set(s) were parsed in 0.02s.
@> 3190 atoms and 1 coordinate set(s) were parsed in 0.02s.


[7Z2O_IAJ] 40 poses, best +0.26
Processing 1 of 1 batches (2 sequences)


@> 7508 atoms and 1 coordinate set(s) were parsed in 0.05s.
@> 7508 atoms and 1 coordinate set(s) were parsed in 0.04s.


[8DSC_NCA] 40 poses, best -0.28
Scored 430 poses of 12/12 complexes
Complexes 12

Top-1 over 12 complexes

     5  re-scored, 41.7%
     2  as DiffDock ranked them, 16.7%
        a random pick off the ensemble, 19.2%


([{'name': '5SAK_ZRY',
   'source': 'rank15_confidence-0.56.sdf',
   'rank_docked': 15,
   'confidence_docked': -0.56,
   'rank_minimised': 1,
   'confidence_minimised': -0.10218106955289841,
   'rmsd': 6.095162315403184},
  {'name': '5SAK_ZRY',
   'source': 'rank4_confidence-0.28.sdf',
   'rank_docked': 4,
   'confidence_docked': -0.28,
   'rank_minimised': 2,
   'confidence_minimised': -0.1706652045249939,
   'rmsd': 4.884946745928307},
  {'name': '5SAK_ZRY',
   'source': 'rank20_confidence-0.68.sdf',
   'rank_docked': 20,
   'confidence_docked': -0.68,
   'rank_minimised': 3,
   'confidence_minimised': -0.2363647222518921,
   'rmsd': 5.964822900602795},
  {'name': '5SAK_ZRY',
   'source': 'rank28_confidence-0.74.sdf',
   'rank_docked': 28,
   'confidence_docked': -0.74,
   'rank_minimised': 4,
   'confidence_minimised': -0.24718543887138367,
   'rmsd': 6.429237241003875},
  {'name': '5SAK_ZRY',
   'source': 'rank1_confidence-0.26.sdf',
   'rank_docked': 1,
   'confidence_docked': -0

## Encoding

`EncodeProtein` solves the RHF for the cutout (runtime: hours), determines its active space with AVAS, takes the 50 orbitals with the most fractional occupancy (strong correlation), as calculated with MP2, then reduces the space further with SHCI. Finally, the compressed active space can be encoded as a Hamiltonian for VQE, or classically solved with CASCI.

`SolveLigand` does not treat the ligand as a quantum-mechanical system; it only solves its RHF.